# Gold: clusters de conteúdo (`governor_clusters`)

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Diagnóstico de `governor_clusters` (saída de `src/modeling/clustering.py`), granularidade de **um
post/reel por linha**. `cluster_label == -1` é ruído do DBSCAN, não um grupo real -- o vocabulário do
produto (`CONTEXT.md`) chama isso de "casos atípicos / virais" na interface final, nunca "-1" cru;
aqui mantemos o rótulo técnico, mas vale lembrar o significado ao interpretar.

Notebook estritamente leitura via `DeltaRepository`, mesmo princípio da ADR
[0003](../../docs/adr/0003-desacoplar-modelagem-do-notebook-via-scripts-cli-com-checkpoint.md).

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from deltalake import DeltaTable
from dotenv import load_dotenv

from config import settings
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import (
    completeness_summary,
    count_duplicate_rows,
    with_governor_metadata,
)

load_dotenv()
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

## 1. Carga + schema

In [ ]:
df_clusters = repo.load_clusters()
df_clusters.dtypes

## 2. Completude

In [ ]:
completude = completeness_summary(df_clusters)
print(f"linhas duplicadas (por id_reel): {count_duplicate_rows(df_clusters, subset=['id_reel'])}")
completude[completude['n_nulos'] > 0]

## 3. Distribuição: tamanho de cada cluster, por tipo de conteúdo

Feed (posts estáticos) e reel são clusterizados separadamente (`src/modeling/clustering.py`) --
comparar contagem por `content_type` evita ler um cluster de vídeo como se fosse do mesmo espaço
que um cluster de post estático.

In [ ]:
pd.crosstab(df_clusters['cluster_label'], df_clusters['content_type'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_clusters['cluster_score'], kde=False, ax=ax, bins=20)
ax.set_title('Distribuição de cluster_score (score de ajuste do algoritmo)')
plt.tight_layout()
plt.show()
print(df_clusters.groupby('cluster_algo')['cluster_score'].describe())

## 4. Evolução temporal

Sem tabela `_history` irmã para `governor_clusters` hoje -- seção do esqueleto padrão pulada de
propósito.

## 5. Relação com covariáveis (partido/UF)

`governor_clusters` só tem `ownerUsername`, não `inputUrl` -- diferente das outras tabelas Gold
deste conjunto de notebooks. Ponte via `profiles_clean` (Silver, tem os dois) antes de juntar com
`governors_metadata`.

In [ ]:
mapa_username_inputurl = DeltaTable(str(settings.SILVER_PROFILES)).to_pandas()[['username', 'inputUrl']]
df_clusters_com_url = df_clusters.merge(
    mapa_username_inputurl, left_on='ownerUsername', right_on='username', how='left'
)
taxa_match = df_clusters_com_url['inputUrl'].notna().mean() * 100
print(f'{taxa_match:.1f}% das linhas de governor_clusters encontraram inputUrl em profiles_clean')

df_clusters_metadado = with_governor_metadata(df_clusters_com_url, repo.load_governors_metadata())
pd.crosstab(df_clusters_metadado['partido'], df_clusters_metadado['cluster_label'], normalize='index').mul(100).round(1)

## 6. Outliers

Posts marcados `cluster_label == -1` ("casos atípicos / virais", no vocabulário do produto) e
linhas sem `inputUrl` encontrado no passo anterior (governador que não bateu com `profiles_clean` --
merece checar se é um `ownerUsername` desatualizado, não um bug silencioso).

In [ ]:
atipicos = df_clusters[df_clusters['cluster_label'] == -1]
print(f'{len(atipicos)} posts em cluster_label == -1')
display(atipicos[['ownerUsername', 'content_type', 'cluster_algo', 'cluster_score']])

sem_match = df_clusters_com_url[df_clusters_com_url['inputUrl'].isna()]
print(f'{len(sem_match)} linhas sem inputUrl correspondente em profiles_clean')
sem_match['ownerUsername'].value_counts()

## Nota de interpretação

`cluster_score` vem do algoritmo inteiro (DBSCAN/Agglomerative), não por post individual -- valores
praticamente idênticos entre linhas do mesmo `content_type`/execução são esperados, não um sinal de
pouca variância real nos dados. O que varia de fato é `cluster_label`.